# Region quantification on Google Colab

Colab runs on a remote VM with **no display**, so the native-window drawer
(`RegionDrawer`, `%matplotlib osx/tk/qt`) cannot work here. This notebook draws
regions in the **browser** on an HTML canvas instead (Colab-only, uses
`google.colab.kernel.invokeFunction`).

Everything else - `load_regions`, `show_regions`, `apply_regions`, `quantify` -
is the same piepy code as the desktop notebook.

In [ ]:
# install piepy + deps on the Colab VM (adjust to your source)
# !pip install git+https://github.com/kaancet/piepy
# from google.colab import drive; drive.mount('/content/drive')  # if data is on Drive

In [ ]:
import numpy as np
from IPython.display import display, HTML
from piepy.sensory.visual.visualSession import VisualSession
from piepy.imaging.onep.widefield.onepAnalysis import OnePAnalysis
from piepy.imaging.onep.widefield.regions import (
    reference_frame,
    load_regions,
    show_regions,
    apply_regions,
    quantify,
)

In [ ]:
vis_sesh = VisualSession('240207_KC149__1P_KC', load_flag=False)
run = vis_sesh.runs[0]
o = OnePAnalysis(run.data.data, run.paths.onepcam)
trial_avg = o.trial_avg(batch_count=3)  # (N, 1, H, W)

## 1. Draw regions in the browser (Colab-native)

Run the next cell. On the canvas: click to add polygon vertices, type a name,
click **Add region**. Repeat per region. Click **Done** to send them to the
kernel (also writes `rois.json`). Regions must be mutually exclusive.

In [ ]:
def _png_b64(img):
    """2D array -> base64 PNG (grayscale, contrast-stretched)."""
    import base64, io
    from PIL import Image
    a = np.nan_to_num(np.asarray(img, dtype=float))
    a -= a.min()
    m = a.max()
    if m:
        a = a / m * 255.0
    buf = io.BytesIO()
    Image.fromarray(a.astype("uint8")).save(buf, "PNG")
    return base64.b64encode(buf.getvalue()).decode()


_HTML = """
<div id="prd"></div>
<script>
(function(){
  const IMG="data:image/png;base64,__IMG__";
  const MAXW=__MAXW__;
  const root=document.getElementById("prd"); root.innerHTML="";
  const status=document.createElement("div"); status.textContent="click image to add vertices";
  const nameIn=document.createElement("input"); nameIn.placeholder="region name"; nameIn.style.margin="4px";
  const img=new Image();
  img.onload=function(){
    const scale=Math.min(1, MAXW/img.width);
    const W=Math.round(img.width*scale), H=Math.round(img.height*scale);
    const cv=document.createElement("canvas"); cv.width=W; cv.height=H;
    cv.style.border="1px solid #888"; cv.style.cursor="crosshair"; cv.style.display="block";
    const ctx=cv.getContext("2d");
    let cur=[]; const saved=[];
    function draw(){
      ctx.clearRect(0,0,W,H); ctx.drawImage(img,0,0,W,H); ctx.lineWidth=2;
      for(const s of saved){
        ctx.strokeStyle="red"; ctx.fillStyle="rgba(255,0,0,0.2)";
        ctx.beginPath(); s.disp.forEach((p,i)=> i?ctx.lineTo(p[0],p[1]):ctx.moveTo(p[0],p[1]));
        ctx.closePath(); ctx.fill(); ctx.stroke();
        const cx=s.disp.reduce((a,p)=>a+p[0],0)/s.disp.length, cy=s.disp.reduce((a,p)=>a+p[1],0)/s.disp.length;
        ctx.fillStyle="red"; ctx.font="14px sans-serif"; ctx.fillText(s.name,cx,cy);
      }
      ctx.strokeStyle="yellow"; ctx.fillStyle="yellow";
      ctx.beginPath(); cur.forEach((p,i)=> i?ctx.lineTo(p[0],p[1]):ctx.moveTo(p[0],p[1])); ctx.stroke();
      cur.forEach(p=>{ctx.beginPath(); ctx.arc(p[0],p[1],3,0,7); ctx.fill();});
    }
    cv.onclick=function(e){ const r=cv.getBoundingClientRect(); cur.push([e.clientX-r.left, e.clientY-r.top]); draw(); };
    function mk(t,fn){const b=document.createElement("button"); b.textContent=t; b.style.margin="4px"; b.onclick=fn; return b;}
    const bAdd=mk("Add region",function(){
      if(cur.length<3){status.textContent="need >=3 vertices"; return;}
      const nm=(nameIn.value||"").trim(); if(!nm){status.textContent="type a name first"; return;}
      saved.push({name:nm, disp:cur.slice()}); cur=[]; nameIn.value=""; status.textContent="added "+nm; draw();
    });
    const bUndo=mk("Undo point",function(){ cur.pop(); draw(); });
    const bDone=mk("Done",function(){
      const out={}; for(const s of saved){ out[s.name]=s.disp.map(p=>[p[0]/scale, p[1]/scale]); }
      status.textContent="sent "+Object.keys(out).length+" regions to kernel";
      google.colab.kernel.invokeFunction("piepy.recv_regions",[JSON.stringify(out)],{});
    });
    root.appendChild(status); root.appendChild(nameIn); root.appendChild(cv);
    root.appendChild(bAdd); root.appendChild(bUndo); root.appendChild(bDone);
    draw();
  };
  img.src=IMG;
})();
</script>
"""


def colab_draw_regions(ref_img, out_json="rois.json", max_width=900):
    """Browser-canvas polygon drawer for Colab. Writes vertices to out_json
    (image-pixel coords) when you click Done; also returns them via the module
    dict COLAB_REGIONS. Then use load_regions(out_json)."""
    import json
    from google.colab import output

    def _recv(js):
        d = json.loads(js)
        COLAB_REGIONS.clear()
        COLAB_REGIONS.update({k: np.asarray(v, float) for k, v in d.items()})
        with open(out_json, "w") as f:
            json.dump(d, f, indent=2)
        return {}

    output.register_callback("piepy.recv_regions", _recv)
    html = _HTML.replace("__IMG__", _png_b64(ref_img)).replace("__MAXW__", str(int(max_width)))
    display(HTML(html))
    print(f"draw, click Done -> writes {out_json}; then run the next cell")


COLAB_REGIONS = {}

In [ ]:
colab_draw_regions(reference_frame(trial_avg))   # or reference_frame(trial_avg, source='anat.tif')

## 2. Load + confirm
`show_regions` renders inline (no GUI needed), so it works on Colab.

In [ ]:
regions = load_regions('rois.json')
show_regions(trial_avg, regions);

## 3. Mask movie -> K masked movies `(N, H, W)`

In [ ]:
masked = apply_regions(trial_avg, regions)   # dict{name: (N, H, W)}, NaN outside region
{k: v.shape for k, v in masked.items()}

## 4. Quantify
- `axis='spatial'` -> `(N,)` time-series per region
- `axis='frames'`  -> `(H, W)` map per region
- `stat` = name or a callable `func(arr, axis, **kwargs)`

In [ ]:
ts_mean = quantify(masked, 'mean', axis='spatial')
map_mean = quantify(masked, 'mean', axis='frames')

# custom function with keyword passthrough
p90 = quantify(masked, np.nanpercentile, axis='frames', q=90)

## 5. Quick look

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
for name, y in ts_mean.items():
    ax.plot(y, label=name)
ax.set_xlabel('frame'); ax.set_ylabel('mean activity'); ax.legend()